## Import datasets and create data format yolo

In [2]:
from datasets import load_dataset
import shutil
import os
import json
ds = load_dataset("keremberke/license-plate-object-detection", "full")

d:\Asus\Ungdung\Miniconda\workspace\envs\AIOEx\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating test split: 100%|██████████| 882/882 [00:00<00:00, 21610.78 examples/s]


In [3]:
def get_img_paths_info(folder_dir=''):
    image_folder = os.path.join(folder_dir, 'image')
    info_folder = os.path.join(folder_dir, 'info')
    json_files = os.listdir(info_folder)[:10]
    image_paths = os.listdir(image_folder)[:10]
    data_list = []
    for i, json_file in enumerate(json_files):
        json_path = os.path.join(info_folder, json_file)
        image_path = os.path.join(image_folder, image_paths[i])
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            img_size = [data['width'], data['height']]
            bboxs = data['objects']['bbox']
        data_list.append((image_path, img_size, bboxs))
    return data_list


# Ví dụ sử dụng:
save_dir = "../license-plate-dataset"
result = get_img_paths_info(os.path.join(save_dir, "train"))
print("Kết quả:", result)

Kết quả: [('../license-plate-dataset\\train\\image\\0.png', [472, 303], [[185.0, 231.0, 107.0, 37.0]]), ('../license-plate-dataset\\train\\image\\1.png', [472, 303], [[172.0, 95.0, 96.0, 83.0]]), ('../license-plate-dataset\\train\\image\\10.png', [472, 303], [[239.0, 139.0, 98.0, 30.0]]), ('../license-plate-dataset\\train\\image\\100.png', [472, 303], [[192.0, 34.0, 86.0, 63.0]]), ('../license-plate-dataset\\train\\image\\1000.png', [472, 303], [[156.0, 115.0, 99.0, 84.0]]), ('../license-plate-dataset\\train\\image\\1001.png', [472, 303], [[317.0, 179.0, 80.0, 53.0]]), ('../license-plate-dataset\\train\\image\\1002.png', [472, 303], [[154.0, 120.0, 92.0, 82.0]]), ('../license-plate-dataset\\train\\image\\1003.png', [472, 303], [[196.0, 101.0, 103.0, 81.0]]), ('../license-plate-dataset\\train\\image\\1004.png', [472, 303], [[215.0, 240.0, 99.0, 25.0]]), ('../license-plate-dataset\\train\\image\\1005.png', [450, 290], [[144.0, 210.0, 65.0, 49.0]])]


In [6]:
train_data = get_img_paths_info('../license-plate-dataset/train')
val_data = get_img_paths_info('../license-plate-dataset/validation')
test_data = get_img_paths_info('../license-plate-dataset/test')

In [ ]:
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np

# Custom Dataset class
class LicensePlateDataset(Dataset):
    def __init__(self, data_list, transforms=None):
        self.data_list = data_list
        self.transforms = transforms

    def __getitem__(self, idx):
        img_path, img_size, bboxes = self.data_list[idx]
        
        # Load image
        image = Image.open(img_path).convert("RGB")
        
        # Convert bboxes to tensor format
        boxes = torch.as_tensor(bboxes, dtype=torch.float32)
        labels = torch.ones((len(bboxes),), dtype=torch.int64)  # all boxes are license plates (class 1)
        
        target = {}
        target["boxes"] = boxes
        target["labels"] = labels
        
        if self.transforms is not None:
            image = self.transforms(image)
        
        return image, target

    def __len__(self):
        return len(self.data_list)

# Create data transforms
data_transforms = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
])

# Create datasets
train_dataset = LicensePlateDataset(train_data, data_transforms)
val_dataset = LicensePlateDataset(val_data, data_transforms)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))

# Create model
model = fasterrcnn_resnet50_fpn(pretrained=True)
num_classes = 2  # background + license plate
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

# Move model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# Set up optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0005)


In [ ]:
best_model_wts = None
best_val_loss = float('inf')

# Training and validation loop
num_epochs = 10
for epoch in range(num_epochs):
    # Training phase
    model.train()
    total_train_loss = 0

    for images, targets in train_loader:
        images = [image.to(device) for image in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        total_train_loss += losses.item()

    # Validation phase
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for images, targets in val_loader:
            images = [image.to(device) for image in images]
            targets = [{k: v.to(device) for k, v in t.items()}
                       for t in targets]

            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            total_val_loss += losses.item()

    # Calculate average losses
    avg_train_loss = total_train_loss / len(train_loader)
    avg_val_loss = total_val_loss / len(val_loader)

    # Print epoch statistics
    print(
        f'Epoch {epoch+1}/{num_epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}')

    # Save the best model
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_model_wts = model.state_dict()

# Load the best model weights
if best_model_wts is not None:
    model.load_state_dict(best_model_wts)

# Save the best model
torch.save(model.state_dict(), 'best_license_plate_detector.pth')